In [1]:

# ============================================================
# Shared setup (anonymized; paths are never printed)
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr

# Grayscale, publication style; no colour, no titles/captions in saved images.
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"image.cmap": "gray", "axes.grid": True, "grid.alpha": 0.25,
                     "font.size": 11, "savefig.dpi": 600})

DATA = Path("../data"); FIG = Path("../results/figures"); TAB = Path("../results/tables")
RNG = np.random.default_rng(42)

# Dataset registry (public sources) and decision domain.
DATASETS = {"telco": "telco_churn.csv", "adult": "adult.parquet",
            "credit_risk": "credit_risk.csv", "gmsc": "gmsc.parquet",
            "taiwan": "taiwan.parquet"}
DOMAIN = {"telco": "non-credit", "adult": "non-credit",
          "credit_risk": "consumer credit", "gmsc": "consumer credit",
          "taiwan": "consumer credit"}
# Grayscale shade + marker per domain (colour-free, print-safe).
STYLE = {"consumer credit": ("0.15", "o"), "corporate credit": ("0.0", "*"),
         "non-credit": ("0.45", "s")}
# Paper-1 anchor (Polish corporate bankruptcy; values reported in the first paper).
POLISH = {"gini": 2*0.778-1, "pbar": 0.0482, "naive_gap": 2.72, "indep_gap": 0.33}

def load(name):
    f = DATA / DATASETS[name]
    return pd.read_parquet(f) if str(f).endswith(".parquet") else pd.read_csv(f)

def prep(name, df):
    # Return numeric design matrix, binary outcome, and two pd-independent drivers (b1,b2).
    df = df.copy()
    if name == "telco":
        y = (df["Churn"].astype(str).str.strip() == "Yes").astype(int).values
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
        X = df.drop(columns=["customerID", "Churn"]); b2col = "TotalCharges"
    elif name == "adult":
        y = df["HIGH_INCOME"].astype(int).values
        drop = ["income", "HIGH_INCOME", "AGE_YEARS", "SEX_STD", "EDU_STD", "VIP", "fnlwgt"]
        X = df.drop(columns=[c for c in drop if c in df]); b2col = "hours-per-week"
    elif name == "credit_risk":
        y = df["loan_status"].astype(int).values
        X = df.drop(columns=["loan_status"]); b2col = "person_income"
    elif name == "gmsc":
        y = df["DEFAULT"].astype(int).values
        X = df.drop(columns=[c for c in ["DEFAULT", "VIP"] if c in df]); b2col = "INCOME"
    elif name == "taiwan":
        y = df["DEFAULT"].astype(int).values
        X = df.drop(columns=[c for c in ["DEFAULT", "VIP", "AGE_YEARS", "SEX_STD"] if c in df]); b2col = "LIMIT_BAL"
    b2 = pd.to_numeric(df[b2col], errors="coerce").values if b2col in df else None
    Xn = pd.get_dummies(X, dummy_na=False).apply(pd.to_numeric, errors="coerce")
    return Xn, y, b2

def system_pd(X, y):
    # L2 logistic system model; out-of-fold PD via 5-fold stratified CV; in-fold median imputation.
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    pdh = np.zeros(len(y)); skf = StratifiedKFold(5, shuffle=True, random_state=42)
    for tr, te in skf.split(X, y):
        Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()
        med = Xtr.median(numeric_only=True); Xtr = Xtr.fillna(med); Xte = Xte.fillna(med)
        sc = StandardScaler().fit(Xtr)
        clf = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)
        clf.fit(sc.transform(Xtr), y[tr]); pdh[te] = clf.predict_proba(sc.transform(Xte))[:, 1]
    return pdh, roc_auc_score(y, pdh)

def pick_b1(X, pdh):
    # Near-orthogonal driver: numeric feature with the lowest |Spearman| vs PD (Attr43 analogue).
    best, br = None, 2.0
    for c in X.columns:
        v = X[c].values.astype(float); m = ~np.isnan(v)
        if m.sum() < 100 or np.nanstd(v) == 0: continue
        r = abs(spearmanr(v[m], pdh[m]).statistic)
        if not np.isnan(r) and r < br: br, best = r, v
    return best

QGRID = [0, 0.10, 0.20, 0.35, 0.55, 0.75, 0.90, 1.0]
def assign_grades(pdh):
    qs = np.quantile(pdh, QGRID); qs[0] -= 1e-9; qs[-1] += 1e-9
    return np.digitize(pdh, qs[1:-1])
SCHED = np.array([0.10, 0.15, 0.20, 0.35, 0.50, 0.70, 0.80])  # ord 0 best .. ord 6 worst
OVR = 0.35

def rank01(v):
    return pd.Series(v).rank(pct=True).fillna(0.5).values

def expected_gap(pi, y):
    # Exact expected upgrade-minus-none default-rate gap (percentage points).
    return (np.sum(pi * y) / np.sum(pi) - y.mean()) * 100.0

def sim_once(pi_firm, grade, G, rng):
    # One generative override draw; returns signed grade_diff (negative = upgrade) and masks.
    N = len(grade)
    overridden = rng.random(N) < OVR
    up = overridden & (rng.random(N) < pi_firm)
    down = overridden & ~up
    notch = np.where(rng.random(N) < 0.70, 1, 2)
    gd = np.zeros(N, int); gd[up] = -notch[up]; gd[down] = notch[down]
    final = np.clip(grade + gd, 0, G - 1)
    return final - grade, overridden, up

def save_fig(fig, name):
    # Save grayscale figure as PNG and PDF at 600 dpi; no caption; no path printed.
    fig.savefig(FIG / (name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(FIG / (name + ".pdf"), bbox_inches="tight")
    plt.close(fig)


# 06_leakage_aware_validation.ipynb -- XGBoost nested feature sets, fold balance, SHAP (one dataset)
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import shap
XGB = dict(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
           colsample_bytree=0.8, eval_metric="logloss", random_state=42, n_jobs=0)
def cv_auc(Xv, y, spw):
    skf = StratifiedKFold(5, shuffle=True, random_state=42); a, p = [], []
    for tr, te in skf.split(Xv, y):
        m = XGBClassifier(scale_pos_weight=spw, **XGB)
        m.fit(Xv[tr], y[tr]); pr = m.predict_proba(Xv[te])[:, 1]
        a.append(roc_auc_score(y[te], pr)); p.append(average_precision_score(y[te], pr))
    return np.mean(a), np.mean(p)
auc_rows, fold_rows = [], []
for name in DATASETS:
    df = load(name); X, y, _ = prep(name, df)
    sc = pd.read_parquet(TAB / ("scores_" + name + ".parquet"))
    pdh = sc["pd"].values; grade = sc["grade"].values.astype(int)
    pif = 0.10 + 0.70*rank01(sc["b1"].values)  # leakage-safe b1 override draw
    gd, ovflag, _ = sim_once(pif, grade, 7, np.random.default_rng(1))
    Xf = X.fillna(X.median(numeric_only=True)).values
    spw = (y == 0).sum()/(y == 1).sum()
    base = Xf
    ovr = np.column_stack([Xf, gd, ovflag.astype(int)])
    full = np.column_stack([Xf, gd, ovflag.astype(int), pdh, grade])
    for lab, Xv in [("financial", base), ("financial+override", ovr), ("full(+pd)", full)]:
        a, p = cv_auc(Xv, y, spw)
        auc_rows.append({"dataset": name, "feature_set": lab, "ROC_AUC": round(a, 4), "PR_AUC": round(p, 4)})
    skf = StratifiedKFold(5, shuffle=True, random_state=42)
    for k, (tr, te) in enumerate(skf.split(Xf, y)):
        fold_rows.append({"dataset": name, "fold": k,
                          "train_dr": round(y[tr].mean(), 4), "test_dr": round(y[te].mean(), 4)})
auct = pd.DataFrame(auc_rows); foldt = pd.DataFrame(fold_rows)
auct.to_csv(TAB / "featureset_auc.csv", index=False); foldt.to_csv(TAB / "fold_balance.csv", index=False)
# delta vs financial
piv = auct.pivot(index="dataset", columns="feature_set", values="ROC_AUC")
piv["delta_override"] = (piv["financial+override"] - piv["financial"]).round(4)
piv.to_csv(TAB / "featureset_delta.csv")
# SHAP for one representative dataset (credit_risk), full model
rep_name = "credit_risk"
dfr = load(rep_name); Xr, yr, _ = prep(rep_name, dfr)
scr = pd.read_parquet(TAB / ("scores_" + rep_name + ".parquet"))
gdr, ovr_, _ = sim_once(0.10+0.70*rank01(scr["b1"].values), scr["grade"].values.astype(int), 7, np.random.default_rng(1))
Xrf = Xr.fillna(Xr.median(numeric_only=True)).copy()
Xrf["grade_diff"] = gdr; Xrf["override_flag"] = ovr_.astype(int)
Xrf["pd_system"] = scr["pd"].values; Xrf["grade_ordinal"] = scr["grade"].values
mdl = XGBClassifier(scale_pos_weight=(yr == 0).sum()/(yr == 1).sum(), **XGB).fit(Xrf.values, yr)
idx = np.random.default_rng(0).choice(len(Xrf), size=min(2000, len(Xrf)), replace=False)
sv = shap.TreeExplainer(mdl).shap_values(Xrf.iloc[idx].values)
imp = pd.Series(np.abs(sv).mean(0), index=Xrf.columns).sort_values(ascending=True).tail(10)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"wspace": 0.5})
ax = axes[0]
x = np.arange(len(DATASETS)); w = 0.26; greys = {"financial": "0.2", "financial+override": "0.5", "full(+pd)": "0.78"}
for k, fs in enumerate(["financial", "financial+override", "full(+pd)"]):
    sub = auct[auct.feature_set == fs].set_index("dataset").loc[list(DATASETS)]
    ax.bar(x + (k-1)*w, sub["ROC_AUC"], w, color=greys[fs], edgecolor="k", linewidth=0.5, label=fs)
ax.set_xticks(x); ax.set_xticklabels(list(DATASETS), rotation=20); ax.set_ylim(0.5, 1.0)
ax.set_ylabel("5-fold ROC-AUC"); ax.legend(fontsize=8)
ax = axes[1]
ax.barh(np.arange(len(imp)), imp.values, color="0.4", edgecolor="k", linewidth=0.5)
ax.set_yticks(np.arange(len(imp))); ax.set_yticklabels([(str(t) if len(str(t))<=18 else str(t)[:17]+chr(8230)) for t in imp.index], fontsize=8)
ax.set_xlabel("mean |SHAP| (credit_risk, full model)")
save_fig(fig, "fig06_leakage_validation")
piv


feature_set,financial,financial+override,full(+pd),delta_override
dataset,,,,
adult,0.9290,0.9288,0.9280,-0.0002
credit_risk,0.9448,0.9447,0.9434,-0.0001
gmsc,0.8651,0.8652,0.8643,0.0001
taiwan,0.7817,0.7817,0.7806,0.0000
telco,0.8394,0.8403,0.8366,0.0009
